# 02. 数据建模

职责1-数据建模：建立评估模型，指导业务制定目标。

涵盖：时序预测（ARIMA/Holt-Winters/Prophet）、回归评估、聚类分群。

## 1. 时序预测 — 销量预测与目标设定

In [ ]:
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
from utils.data_loader import load_byd_monthly
from modeling import (ARIMAForecaster, ExponentialSmoothingForecaster, 
                      MovingAverageForecaster, compare_forecasters)

series = load_byd_monthly().set_index('month')['sales']
series

In [ ]:
# 模型对比
compare_forecasters(series, steps=3)

In [ ]:
# ARIMA 预测
res = ARIMAForecaster((1,1,1)).fit_predict(series, steps=3)
print('预测指标:', res.metrics)
print('未来3月预测:')
print(res.forecast)

## 2. 回归评估模型 — 量化影响因素

In [ ]:
from modeling import RegressionEvaluator, compare_models

daily = ds['daily_sales'] if 'ds' in dir() else __import__('utils.data_loader', fromlist=['prepare_all']).prepare_all()['daily_sales']
X = daily[['ad_spend', 'is_weekend', 'is_holiday']]
y = daily['sales']
compare_models(X, y)

In [ ]:
# XGBoost 特征重要性
res = RegressionEvaluator('xgboost').fit(X, y)
print(res.summary())
res.feature_importance.plot(kind='barh', title='特征重要性')

## 3. 聚类分析 — 产品/区域分群

In [ ]:
from modeling import ClusterAnalyzer
md = ds['multi_dim_sales']
profile = md.groupby('model').agg({'sales': ['sum', 'mean', 'std']}).round(0)
profile.columns = ['total', 'mean', 'std']
profile = profile.fillna(0)

# 选 K
sil = ClusterAnalyzer.silhouette_at_k(profile.reset_index(), ['total','mean','std'])
print(sil)

# K=3 聚类
clustered = ClusterAnalyzer('kmeans', n_clusters=3).fit(profile.reset_index(), ['total','mean','std'])
print(clustered[['model','cluster']])

## 小结

- 时序预测辅助业务设定月度/季度目标
- 回归模型量化广告投放等因素的贡献
- 聚类识别产品线结构，支持差异化目标管理